# Demo 03 : Middleware hooks

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · middleware-hooks

We observe the agent lifecycle and the model and tool calls.

In [2]:
import json
import re
import sqlite3
from pathlib import Path

from langchain.tools import tool

DB = data("game-studio/studio.db")
SECTIONS = []
for path in sorted(Path(data_folder("game-studio/docs")).glob("*.md")):
    title, *parts = path.read_text(encoding="utf-8").split("\n## ")
    SECTIONS += [f"{title.splitlines()[0]}\n## {part}" for part in parts]


def rows(sql, *args):
    with sqlite3.connect(DB) as conn:
        conn.row_factory = sqlite3.Row
        return [dict(row) for row in conn.execute(sql, args)]


def words(text):
    return set(re.findall(r"[a-z0-9-]{3,}", text.lower()))


def best(texts, query, k):
    q = words(query)
    return sorted(texts, key=lambda t: -len(q & words(t)))[:k]

@tool
def game_info(title: str) -> str:
    """Look up a Comet Arcade game by its title or id. Returns id, title, genre, status and pitch."""
    t = title.strip()
    found = rows("SELECT game_id, title, genre, status, pitch FROM games WHERE game_id = ? OR title LIKE ?", t.upper(), f"%{t}%")
    return json.dumps(found[0]) if found else "Unknown game"

@tool
def assets_for_game(game_id: str) -> str:
    """List sprites, palettes and sounds for a game id such as CS-01, with colour and status."""
    return json.dumps(rows("SELECT asset_id, kind, name, colour, status FROM assets WHERE game_id = ?", game_id.strip().upper()))

@tool
def search_design_docs(query: str) -> str:
    """Search the design docs and the studio style guide. Returns the three best matching sections."""
    return "\n\n---\n\n".join(best(SECTIONS, query, 3))

tools = [game_info, assets_for_game, search_design_docs]
question = {"messages": [("human", "Playtesters lose Comet Snake to comets they cannot see. Look up the game first, then its assets, "
                                   "then the style guide section on the rule that applies. Which asset is at fault, and what should the Artist change?")]}


def show(result):
    for message in result["messages"]:
        print(message.type, str(getattr(message, "tool_calls", None) or message.content)[:150].replace("\n", " "))
    print("model calls:", sum(m.type == "ai" for m in result["messages"]))
    print("input tokens per call:", [m.usage_metadata["input_tokens"] for m in result["messages"] if m.type == "ai"])

We log the lifecycle and each model and tool call. Hooks observe or update state, while wrappers control whether the handler runs. Each hook adds a graph step, so this agent needs a larger `recursion_limit` than the plain one.

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, wrap_tool_call


class Logger(AgentMiddleware):
    def before_agent(self, state, runtime):
        print("agent: start")

    def before_model(self, state, runtime):
        print("model input messages:", len(state["messages"]))

    def after_model(self, state, runtime):
        print("requested tools:", [c["name"] for c in state["messages"][-1].tool_calls])

    def after_agent(self, state, runtime):
        print("agent: end")


@wrap_tool_call
def log_tool(request, handler):
    print("tool:", request.tool_call["name"], request.tool_call["args"])
    result = handler(request)
    print("result:", result.content[:100].replace("\n", " "))
    return result


agent = create_agent(llm, tools=tools, middleware=[Logger(), log_tool],
                     system_prompt="You are the studio assistant of Comet Arcade. Use the tools. Answer in three sentences.")
out = agent.invoke(question, {"recursion_limit": 30})
print("answer:", out["messages"][-1].content)

agent: start
model input messages: 1


requested tools: ['game_info']
tool: game_info {'title': 'Comet Snake'}
result: {"game_id": "CS-01", "title": "Comet Snake", "genre": "arcade", "status": "in development", "pitch":
model input messages: 3


requested tools: ['assets_for_game']
tool: assets_for_game {'game_id': 'CS-01'}
result: [{"asset_id": "CS-SPR-01", "kind": "sprite", "name": "snake head", "colour": "#7cf5c4", "status": "f
model input messages: 5


requested tools: ['search_design_docs']
tool: search_design_docs {'query': 'style guide rule for sprite visibility and contrast against background, especially hazards/comets; Artist change asset colors contrast'}
result: # Comet Arcade style guide ## Contrast  Anything that can end the game needs a contrast of at least 
model input messages: 7


requested tools: []
agent: end
answer: The at-fault asset is **CS-SPR-03, the comet sprite**: its draft color `#3a4a7a` has too little contrast against the deep-space background `#0b1030`. The style guide requires game-ending hazards to meet at least 4.5:1 contrast; the Artist should keep the comet’s shape and change its color to the studio danger color, **`#ff6b5e`**.


We keep logging outside the prompt. Logs can contain identifiers and tool data. We limit access and redact sensitive fields before recording real traffic.

## Things to try

Optional : we can change the game, the asset or the question and run the example again.